# ETL y Limpieza de Datos Financieros y Macroeconómicos

Pipeline de **Extracción, Transformación y Carga (ETL)** para procesar datasets macroeconómicos de `data/raw/` y guardarlos estandarizados en `data/processed/`.

In [19]:
import sys
import re
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np

NOTEBOOKS_DIR = Path.cwd()
ROOT_DIR = NOTEBOOKS_DIR.parent if NOTEBOOKS_DIR.name == "notebooks" else NOTEBOOKS_DIR
RAW_DATA_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raíz: {ROOT_DIR}")
print(f"Raw: {RAW_DATA_DIR}")
print(f"Processed: {PROCESSED_DATA_DIR}")

Raíz: c:\Users\ferna\Desktop\tpm-chile
Raw: c:\Users\ferna\Desktop\tpm-chile\data\raw
Processed: c:\Users\ferna\Desktop\tpm-chile\data\processed


# Funciones Auxiliares de Limpieza y Diagnóstico

In [20]:
def normalizar_texto(texto: str) -> str:
    if not isinstance(texto, str):
        return texto
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = texto.strip().lower()
    texto = re.sub(r"[^\w\s]", "", texto)
    return re.sub(r"\s+", "_", texto)

def inspeccionar_dataframe(df: pd.DataFrame, nombre: str):
    print(f"\n--- DIAGNÓSTICO: {nombre} ---")
    print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
    print(f"Duplicados: {df.duplicated().sum()}")
    info_df = pd.DataFrame({
        "Dtype": df.dtypes,
        "Nulos": df.isna().sum(),
        "Unicos": df.nunique()
    })
    print(info_df)
    print(df.head(2))
    print("-" * 35)

# Pipeline Modular de Limpieza por Dataset

In [21]:
def cargar_y_limpiar_excel(ruta_archivo: Path) -> pd.DataFrame:
    nombre_base = ruta_archivo.stem
    print(f"Procesando: {ruta_archivo.name}...")
    
    # 1. Cargar archivo crudo sin asumir encabezado fijo
    df_raw = pd.read_excel(ruta_archivo, header=None)
    
    # 2. Localizar la fila del encabezado real ('periodo', 'fecha' o 'date')
    header_idx = None
    keywords = ["periodo", "fecha", "date"]
    
    for idx, row in df_raw.iterrows():
        row_str = [str(cell).lower() for cell in row.values if pd.notna(cell)]
        if any(kw in cell for cell in row_str for kw in keywords):
            header_idx = idx
            break
            
    df = pd.read_excel(ruta_archivo, skiprows=header_idx) if header_idx is not None else df_raw.copy()
    
    # 3. La primera columna siempre es 'fecha'; la segunda columna es la variable numérica
    col_fecha = df.columns[0]
    col_valor = df.columns[1]
    
    # Estandarizar valores numéricos (comas a puntos)
    serie_valor = df[col_valor]
    if serie_valor.dtype == "object":
        serie_valor = serie_valor.astype(str).str.replace(",", ".").str.strip()
        
    df_limpio = pd.DataFrame({
        "fecha": pd.to_datetime(df[col_fecha], errors="coerce", dayfirst=True),
        nombre_base: pd.to_numeric(serie_valor, errors="coerce")
    })
    
    # 4. Eliminar fechas nulas y ordenar de forma cronológica estricta
    df_limpio = df_limpio.dropna(subset=["fecha"]).sort_values("fecha").reset_index(drop=True)
    
    inspeccionar_dataframe(df_limpio, nombre_base)
    
    # 5. Aserciones de control de calidad
    assert not df_limpio.empty, f"El DataFrame {nombre_base} quedó vacío tras la limpieza."
    assert df_limpio["fecha"].isna().sum() == 0, f"Hay fechas nulas no procesadas en {nombre_base}."
    
    return df_limpio

# Ejecución Batch y Guardado en 
>data/processed/

In [24]:
archivos_raw = list(RAW_DATA_DIR.glob("*.xlsx")) + list(RAW_DATA_DIR.glob("*.xls"))

if not archivos_raw:
    print("⚠️ No se encontraron archivos Excel en data/raw/")
else:
    resumen_procesamiento = []

    for archivo in archivos_raw:
        df_limpio = cargar_y_limpiar_excel(archivo)
        
        nombre_salida = f"{archivo.stem}_clean.csv"
        ruta_salida = PROCESSED_DATA_DIR / nombre_salida
        
        df_limpio.to_csv(ruta_salida, index=False, encoding="utf-8")
        assert ruta_salida.exists(), f"El archivo {nombre_salida} no se guardó."
        
        resumen_procesamiento.append({
            "Archivo Original": archivo.name,
            "Archivo Procesado": nombre_salida,
            "Filas": df_limpio.shape[0],
            "Columnas": df_limpio.shape[1],
            "Nulos Totales": df_limpio.isna().sum().sum()
        })

    print("\n======== RESUMEN DE PROCESAMIENTO ========")
    df_resumen = pd.DataFrame(resumen_procesamiento)
    print(df_resumen.to_string(index=False))

Procesando: clp_usd.xlsx...

--- DIAGNÓSTICO: clp_usd ---
Dimensiones: 11004 filas × 2 columnas
Duplicados: 0
                  Dtype  Nulos  Unicos
fecha    datetime64[us]      0   11004
clp_usd         float64      0    9936
       fecha  clp_usd
0 1982-08-09    55.65
1 1982-08-10    62.79
-----------------------------------
Procesando: cu_usd_lb.xlsx...

--- DIAGNÓSTICO: cu_usd_lb ---
Dimensiones: 3474 filas × 2 columnas
Duplicados: 0
                    Dtype  Nulos  Unicos
fecha      datetime64[us]      0    3474
cu_usd_lb         float64      0    3049
       fecha  cu_usd_lb
0 2012-10-05       3.76
1 2012-10-08       3.76
-----------------------------------
Procesando: desempleo_total_mensual.xlsx...

--- DIAGNÓSTICO: desempleo_total_mensual ---
Dimensiones: 197 filas × 2 columnas
Duplicados: 0
                                  Dtype  Nulos  Unicos
fecha                    datetime64[us]      0     197
desempleo_total_mensual         float64      0     197
       fecha  desemple

# Verificación de archivos generados

In [25]:
archivos_procesados = list(PROCESSED_DATA_DIR.glob("*_clean.csv"))

print(f"Verificando {len(archivos_procesados)} archivos CSV en {PROCESSED_DATA_DIR}:")
for csv_path in archivos_procesados:
    df_check = pd.read_csv(csv_path)
    assert not df_check.empty, f"El archivo {csv_path.name} está vacío."
    print(f"✓ {csv_path.name}: {df_check.shape[0]} filas × {df_check.shape[1]} columnas")

Verificando 8 archivos CSV en c:\Users\ferna\Desktop\tpm-chile\data\processed:
✓ clp_usd_clean.csv: 11004 filas × 2 columnas
✓ cu_usd_lb_clean.csv: 3474 filas × 2 columnas
✓ desempleo_total_mensual_clean.csv: 197 filas × 2 columnas
✓ exp_tpm_mensual_mediana_clean.csv: 301 filas × 2 columnas
✓ imacec_empalmado_original_clean.csv: 367 filas × 2 columnas
✓ ipc_general_anual_clean.csv: 97 filas × 2 columnas
✓ ipc_general_mensual_clean.csv: 1182 filas × 2 columnas
✓ tpm_bcentral_diaria_clean.csv: 7391 filas × 2 columnas
